# Notebook — Teste do módulo `gnss.py`

Este notebook testa o módulo:

```python
geodesy/gnss.py
```

Serão testadas as funções:

- `geometric_range`
- `pseudorange_model`
- `satellite_elevation_azimuth`
- `simple_tropospheric_delay`
- `gnss_height_to_orthometric`

Os exemplos simulam receptor GNSS, satélites em coordenadas ECEF, pseudodistâncias, elevação/azimute, atraso troposférico e conversão de altura elipsoidal para ortométrica.


In [ ]:
# ============================================================
# 0. CONFIGURAÇÃO INICIAL
# ============================================================

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

current_dir = Path.cwd()

if (current_dir / "geodesy").exists():
    sys.path.insert(0, str(current_dir))
    print("Pasta geodesy encontrada no diretório atual.")
else:
    print("Atenção: a pasta geodesy não foi encontrada no diretório atual.")
    print("Coloque este notebook no mesmo diretório da pasta geodesy/.")

import geodesy
from geodesy import gnss
from geodesy import coordinates
from geodesy import geoid

print("Versão do pacote geodesy:", geodesy.__version__)


## 1. Receptor GNSS em coordenadas geodésicas e ECEF


In [ ]:
# ============================================================
# 1. RECEPTOR
# ============================================================

receiver_lat = -3.2
receiver_lon = -52.2
receiver_h = 120.0

Xr, Yr, Zr = coordinates.geodetic_to_ecef(receiver_lat, receiver_lon, receiver_h)

receiver_xyz = np.array([Xr, Yr, Zr])

print("Receptor geodésico:")
print(receiver_lat, receiver_lon, receiver_h)

print("\nReceptor ECEF:")
print(receiver_xyz)


## 2. Satélites simulados em ECEF


In [ ]:
# ============================================================
# 2. SATÉLITES SIMULADOS
# ============================================================

# Raio orbital aproximado GNSS em relação ao centro da Terra
R_sat = 26560000.0

sat_lats = np.array([20, 35, 5, -15, 50, -40], dtype=float)
sat_lons = np.array([-60, -30, -100, -80, 10, -140], dtype=float)
sat_h = np.full_like(sat_lats, R_sat - 6371000.0)

Xs, Ys, Zs = coordinates.geodetic_to_ecef(sat_lats, sat_lons, sat_h)

sat_xyz = np.vstack([Xs, Ys, Zs])

print("Satélites ECEF:")
for i in range(len(sat_lats)):
    print(i+1, Xs[i], Ys[i], Zs[i])


In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

ax.scatter([Xr], [Yr], [Zr], marker="o", s=60, label="Receptor")
ax.scatter(Xs, Ys, Zs, marker="^", s=60, label="Satélites")

for i in range(len(Xs)):
    ax.plot([Xr, Xs[i]], [Yr, Ys[i]], [Zr, Zs[i]], linestyle="--", alpha=0.4)

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title("Receptor e satélites em ECEF")
ax.legend()
plt.show()


## 3. Distância geométrica


In [ ]:
# ============================================================
# 3. DISTÂNCIA GEOMÉTRICA
# ============================================================

ranges = []

for i in range(len(sat_lats)):
    satellite_xyz = np.array([Xs[i], Ys[i], Zs[i]])
    rho = gnss.geometric_range(receiver_xyz, satellite_xyz)
    ranges.append(rho)

ranges = np.array(ranges)

print("Distâncias geométricas:")
print(ranges)

plt.figure(figsize=(8, 4))
plt.bar([f"S{i+1}" for i in range(len(ranges))], ranges/1000)
plt.ylabel("Distância (km)")
plt.title("Distância geométrica receptor-satélite")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 4. Pseudodistância


In [ ]:
# ============================================================
# 4. PSEUDODISTÂNCIA
# ============================================================

receiver_clock_bias = 2e-7      # segundos
satellite_clock_biases = np.array([1e-8, -2e-8, 0.5e-8, 3e-8, -1e-8, 2e-8])

pseudoranges = []

for i in range(len(sat_lats)):
    satellite_xyz = np.array([Xs[i], Ys[i], Zs[i]])
    pr = gnss.pseudorange_model(
        receiver_xyz,
        satellite_xyz,
        receiver_clock_bias=receiver_clock_bias,
        satellite_clock_bias=satellite_clock_biases[i]
    )
    pseudoranges.append(pr)

pseudoranges = np.array(pseudoranges)

print("Pseudodistâncias:")
print(pseudoranges)

print("\nDiferença pseudodistância - distância geométrica:")
print(pseudoranges - ranges)

plt.figure(figsize=(8, 4))
plt.plot(ranges/1000, marker="o", label="Distância geométrica")
plt.plot(pseudoranges/1000, marker="s", label="Pseudodistância")
plt.xlabel("Satélite")
plt.ylabel("Distância (km)")
plt.title("Distância geométrica vs pseudodistância")
plt.legend()
plt.grid(True)
plt.show()


## 5. Elevação e azimute dos satélites


In [ ]:
# ============================================================
# 5. ELEVAÇÃO E AZIMUTE
# ============================================================

elevations = []
azimuths = []

for i in range(len(sat_lats)):
    elev, az = gnss.satellite_elevation_azimuth(
        receiver_lat,
        receiver_lon,
        receiver_h,
        Xs[i],
        Ys[i],
        Zs[i],
        degrees=True
    )
    elevations.append(elev)
    azimuths.append(az)

elevations = np.array(elevations)
azimuths = np.array(azimuths)

print("Elevações:")
print(elevations)

print("\nAzimutes:")
print(azimuths)

plt.figure(figsize=(8, 4))
plt.bar([f"S{i+1}" for i in range(len(elevations))], elevations)
plt.axhline(0, linestyle="--")
plt.xlabel("Satélite")
plt.ylabel("Elevação (graus)")
plt.title("Elevação dos satélites")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# Gráfico polar de azimute e elevação

theta = np.deg2rad(azimuths)
r = 90 - elevations

plt.figure(figsize=(7, 7))
ax = plt.subplot(111, projection="polar")
ax.scatter(theta, r, s=80)

for i in range(len(theta)):
    ax.text(theta[i], r[i], f"S{i+1}")

ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.set_rlim(90, 0)
ax.set_title("Skyplot GNSS simplificado")
plt.show()


## 6. Atraso troposférico simples


In [ ]:
# ============================================================
# 6. ATRASO TROPOSFÉRICO
# ============================================================

elev_grid = np.linspace(5, 90, 300)
delay = gnss.simple_tropospheric_delay(elev_grid, zenith_delay=2.3)

plt.figure(figsize=(8, 4))
plt.plot(elev_grid, delay)
plt.xlabel("Elevação (graus)")
plt.ylabel("Atraso troposférico (m)")
plt.title("Atraso troposférico simples")
plt.grid(True)
plt.show()

sat_delays = gnss.simple_tropospheric_delay(np.maximum(elevations, 1.0), zenith_delay=2.3)

print("Atrasos dos satélites:")
print(sat_delays)

plt.figure(figsize=(8, 4))
plt.bar([f"S{i+1}" for i in range(len(sat_delays))], sat_delays)
plt.ylabel("Atraso (m)")
plt.title("Atraso troposférico por satélite")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 7. Conversão de altura GNSS para altura ortométrica


In [ ]:
# ============================================================
# 7. h -> H COM GEOIDE
# ============================================================

h_gnss = np.linspace(80, 200, 100)
N_geoid = 18 + 2*np.sin(np.linspace(0, 2*np.pi, len(h_gnss)))

H_ortho = gnss.gnss_height_to_orthometric(h_gnss, N_geoid)

plt.figure(figsize=(9, 4))
plt.plot(h_gnss, label="h GNSS")
plt.plot(N_geoid, label="N")
plt.plot(H_ortho, label="H = h - N")
plt.xlabel("Ponto")
plt.ylabel("Altura (m)")
plt.title("Conversão de altura elipsoidal GNSS para ortométrica")
plt.legend()
plt.grid(True)
plt.show()

print("H ortométrica min/max:", H_ortho.min(), H_ortho.max())


## 8. Mapa sintético de alturas GNSS


In [ ]:
# ============================================================
# 8. MAPA SINTÉTICO DE ALTURAS
# ============================================================

lon = np.linspace(-55, -50, 100)
lat = np.linspace(-5, -1, 80)

LON, LAT = np.meshgrid(lon, lat)

h_map = 120 + 40*np.exp(-((LON+52.5)**2 + (LAT+3.0)**2)/0.8)
N_map = 18 + 3*np.sin((LON+55)*2*np.pi/5)*np.cos((LAT+5)*np.pi/4)

H_map = gnss.gnss_height_to_orthometric(h_map, N_map)

plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
c1 = plt.contourf(LON, LAT, h_map, levels=30)
plt.colorbar(c1, label="h (m)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Altura elipsoidal GNSS")

plt.subplot(1, 3, 2)
c2 = plt.contourf(LON, LAT, N_map, levels=30)
plt.colorbar(c2, label="N (m)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geoide")

plt.subplot(1, 3, 3)
c3 = plt.contourf(LON, LAT, H_map, levels=30)
plt.colorbar(c3, label="H (m)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Altura ortométrica")

plt.tight_layout()
plt.show()


## 9. Mapa global didático em Mollweide


In [ ]:
# ============================================================
# 9. MAPA GLOBAL DE ALTURA ORTOMÉTRICA SINTÉTICA
# ============================================================

import cartopy.crs as ccrs
import cartopy.feature as cfeature

lon_g = np.arange(-180.0, 180.0 + 1.0, 1.0)
lat_g = np.arange(-90.0, 90.0 + 1.0, 1.0)

LON_G, LAT_G = np.meshgrid(lon_g, lat_g)

h_g = 100 + 40*np.cos(np.deg2rad(LAT_G))**2 + 20*np.sin(np.deg2rad(LON_G))
N_g = 20 + 5*np.sin(np.deg2rad(2*LON_G))*np.cos(np.deg2rad(LAT_G))

H_g = gnss.gnss_height_to_orthometric(h_g, N_g)

fig = plt.figure(figsize=(13, 6))
ax = plt.axes(projection=ccrs.Mollweide())

cf = ax.contourf(
    LON_G,
    LAT_G,
    H_g,
    levels=40,
    transform=ccrs.PlateCarree()
)

ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.set_global()

plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.06, label="H (m)")
ax.set_title("Altura ortométrica sintética a partir de GNSS + geoide — Mollweide")
plt.show()


## 10. Resumo final


Se todas as células foram executadas sem erro, o módulo `gnss.py` está funcionando corretamente.

Os próximos notebooks podem testar:

```python
geodesy/networks.py
geodesy/geodynamics.py
```
